# Expected SARSA on the overtaking route

Train on three slower vehicles ahead, then compare frozen transfer in both directions on the 2,250 m / 120 s route. Run All reads the bundled results. Set `RUN_TRAINING = True` for a fresh run.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("SDL_AUDIODRIVER", "dummy")
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")
here = Path.cwd().resolve()
repo = next((p for p in (here, *here.parents) if (p / "runs/index.csv").is_file()), None)
if repo is None:
    raise RuntimeError("Open this notebook from the MORL4AVSO checkout.")
sys.path.insert(0, str(repo))


import json

import pandas as pd
from IPython.display import display

from environments.sensor_highway import ExperimentConfig
from experiments.route_notebook_helpers import (
    evaluate_route, new_run_directory, summarize_trials, train_sarsa_pair,
)

from algorithms.sensor_agent import LinearExpectedSarsa

In [ ]:
RUN_TRAINING = False
TRAIN_EPISODES = None  # None uses the recorded episode budget.
recorded = repo / "runs/training/overtaking_120s_2250m_1000ep_20261122_100614_534843"

config = ExperimentConfig.load(recorded / "config.json")
metadata = json.loads((recorded / "metadata.json").read_text())
protocol = {key: metadata[key] for key in ("settings", "training_seed", "test_seeds", "test_weather")}
if TRAIN_EPISODES is not None:
    protocol["settings"]["episodes"] = TRAIN_EPISODES

## Matched evaluation

Finish time averages successful trips. Sensing power averages all trips.

In [ ]:
if RUN_TRAINING:
    output = new_run_directory(repo, "sarsa_overtaking")
    agents = train_sarsa_pair(config, protocol, output)
    agents.update({"idle_all_on": None, "random_all_on": None})
    trials = evaluate_route(config, agents, protocol)
    trials.to_csv(output / "evaluation.csv", index=False)
    summarize_trials(trials).to_csv(output / "summary.csv")
    print(f"Saved to {output.relative_to(repo)}")
else:
    trials = pd.read_csv(recorded / "evaluation.csv")

display(summarize_trials(trials).round(3))

## Transfer between traffic conditions

In [ ]:
if RUN_TRAINING:
    base = repo / "runs/training/time_completion_120s_1000ep_20261121_204241_241658"
    base_config = ExperimentConfig.load(base / "config.json")
    base_agents = {
        "adaptive": LinearExpectedSarsa.load(base / "adaptive_agent.npz"),
        "learned_all_on": LinearExpectedSarsa.load(base / "always_on_agent.npz"),
    }
    into_overtaking = evaluate_route(config, base_agents, protocol)
    into_original = evaluate_route(base_config, agents, protocol)
    into_overtaking.to_csv(output / "frozen_transfer_evaluation.csv", index=False)
    into_original.to_csv(output / "original_traffic_evaluation.csv", index=False)
else:
    into_overtaking = pd.read_csv(recorded / "frozen_transfer_evaluation.csv")
    into_original = pd.read_csv(recorded / "original_traffic_evaluation.csv")

for label, data in (
    ("Original training → overtaking", into_overtaking),
    ("Overtaking training → original traffic", into_original),
):
    print(label)
    display(summarize_trials(data).round(3))